In [ ]:
ngkah 1

# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


* Credit Score Classification

Dataset ini berisi data historis keuangan nasabah (seperti pendapatan tahunan, jumlah pinjaman, riwayat keterlambatan bayar, dll).
Tujuan Machine Learning: Membangun model klasifikasi untuk memprediksi Credit Score (Skor Kredit) nasabah ke dalam kategori seperti Poor, Standard, atau Good. Model ini akan sangat berguna bagi bank atau institusi keuangan untuk mengotomatiskan proses evaluasi kelayakan kredit secara cepat dan objektif.

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Semua library dasar berhasil di-import!")

Semua library dasar berhasil di-import!


# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [4]:
# Ganti 'credit_score_dataset.csv' dengan nama file asli dataset Anda
dataset_path = 'credit_score_dataset.csv'

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset berhasil dimuat! Total baris: {df.shape[0]}, Total kolom: {df.shape[1]}")

    # Menampilkan 5 data teratas untuk memeriksa kesesuaian struktur
    display(df.head())
else:
    print("File tidak ditemukan! Pastikan Anda sudah mengunggah file csv ke runtime Colab.")

File tidak ditemukan! Pastikan Anda sudah mengunggah file csv ke runtime Colab.


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# 1. Melihat informasi umum (tipe data dan nilai non-null)
print("--- INFO DATASET ---")
df.info()

# 2. Mengecek jumlah missing values (nilai kosong)
print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# 3. Statistik deskriptif untuk kolom numerik
display(df.describe())

# 4. Visualisasi Distribusi Target (Pastikan nama kolom target sesuai, misalnya 'Credit_Score')
# Ganti 'Credit_Score' dengan nama kolom target di CSV Anda jika berbeda.
target_col = 'Credit_Score'

if target_col in df.columns:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=target_col, palette='viridis')
    plt.title('Distribusi Kelas Target (Credit Score)')
    plt.show()
else:
    print(f"Kolom target '{target_col}' tidak ditemukan. Mohon sesuaikan namanya.")

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [3]:
# Hapus duplikat
df_clean = df.drop_duplicates()

# Pisahkan fitur (X) dan target (y)
# Ganti 'Credit_Score' dengan kolom target Anda
target_col = 'Credit_Score'
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# Identifikasi kolom numerik dan kategorikal secara otomatis
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Buat Pipeline Preprocessing
# 1. Numerik: Isi nilai kosong dengan median, lalu standarisasi (StandardScaler)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Kategorikal: Isi nilai kosong dengan modus, lalu One-Hot Encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Gabungkan pipeline menggunakan ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit dan Transform data
X_processed = preprocessor.fit_transform(X)

# Dapatkan nama kolom baru hasil One-Hot Encoding
cat_encoded_cols = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)
all_cols = num_cols + list(cat_encoded_cols)

# Jadikan DataFrame kembali dan gabungkan dengan target
df_processed = pd.DataFrame(X_processed, columns=all_cols)
df_processed[target_col] = y.reset_index(drop=True)

# Simpan objek preprocessor (Wajib untuk tahap serving/inference nanti)
joblib.dump(preprocessor, 'preprocessor.pkl')

# Simpan dataset yang sudah bersih
df_processed.to_csv('data_preprocessed.csv', index=False)

print("Preprocessing Selesai! Data bersih disimpan sebagai 'data_preprocessed.csv'")
print(f"Bentuk data setelah preprocessing: {df_processed.shape}")
display(df_processed.head())

NameError: name 'df' is not defined